In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, brier_score_loss, confusion_matrix
from sklearn.model_selection import train_test_split

training_dataset = pd.read_csv("encoded_training_table.csv")

## Chronological Splitting

In [30]:
target = training_dataset['responded']
personal_model_dataset = training_dataset.copy()
personal_model_dataset = personal_model_dataset.drop(columns= ['uuid','responded'])

split_index = int(len(training_dataset) * 0.80)

personal_model_X_train = personal_model_dataset.iloc[:split_index]
personal_model_y_train = target.iloc[:split_index]
personal_model_X_test = personal_model_dataset.iloc[split_index:]
personal_model_y_test = target.iloc[split_index:]

train = training_dataset.copy().iloc[:split_index] 
test = training_dataset.copy().iloc[split_index:]

# general_model_X_train, general_model_X_test, general_model_y_train, general_model_y_test = train_test_split(
#     general_model_dataset, target, test_size=0.2, random_state=42, stratify=target
# )

## Data Scaling

In [31]:
scale_cols = [
    'battery_level',
    'light_level',
    'noise_level',
    'acc_avg',
    'notifications_30m',
    'app_events_30m',
    'unlocks_30m',
    'prev_notifications',
    'prev_response_rate',
    'recent_response_rate',
    'slot_prev_response_rate',
    'mins_since_prev_notification',
    'age'
]

scaler = StandardScaler()
personal_model_X_train_scaled = personal_model_X_train.copy()
personal_model_X_test_scaled = personal_model_X_test.copy()

personal_model_X_train_scaled[scale_cols] = scaler.fit_transform(personal_model_X_train[scale_cols])
personal_model_X_test_scaled[scale_cols] = scaler.transform(personal_model_X_test[scale_cols])

## Logistic Regression Model Training

In [32]:
personal_model = LogisticRegression(max_iter=2000, random_state=42)
personal_model.fit(personal_model_X_train_scaled, personal_model_y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [33]:
# y_predictions = general_model.predict(general_model_X_test_scaled)
y_probability = personal_model.predict_proba(personal_model_X_test_scaled)[:, 1]
y_predictions = (y_probability >= 0.43).astype(int)

print("Accuracy:", round(accuracy_score(personal_model_y_test, y_predictions), 3))
print("Precision:", round(precision_score(personal_model_y_test, y_predictions, zero_division=0), 3))
print("Recall:", round(recall_score(personal_model_y_test, y_predictions, zero_division=0), 3))
print("F1-score:", round(f1_score(personal_model_y_test, y_predictions, zero_division=0), 3))
# print("ROC-AUC:", round(roc_auc_score(personal_model_y_test, y_probability), 3))
# print("Log loss:", round(log_loss(personal_model_y_test, y_probability), 3))
print("Brier score:", round(brier_score_loss(personal_model_y_test, y_probability), 3))
print("Confusion Matrix:")
print(confusion_matrix(personal_model_y_test, y_predictions))

Accuracy: 0.794
Precision: 0.497
Recall: 0.446
F1-score: 0.47
Brier score: 0.137
Confusion Matrix:
[[15864  2096]
 [ 2569  2072]]


In [29]:
time_slots = [
    "early morning",
    "mid morning",
    "late morning",
    "afternoon",
    "evening",
    "night"
]

results = []

for slot in time_slots:

    # Copy the same user context.
    candidate = personal_model_X_test_scaled.copy()

    # Reset all time-slot columns to 0.
    for name in time_slots[1:]:
        candidate[f"time_slot_{name}"] = 0

    # Activate the selected time slot.
    if(slot != "early morning"):
        candidate[f"time_slot_{slot}"] = 1

    # Predict the probability of responding.
    probabilities = personal_model.predict_proba(candidate)[1, 1]

    results.append({
        "time_slot": slot,
        "average_response_probability": round(probabilities.mean() * 100, 2)
    })

# Display results from highest to lowest probability.
results = pd.DataFrame(results).sort_values(
    "average_response_probability",
    ascending=False
)

print(results.to_string(index=False))

print(
    "\nRecommended time slot:",
    results.iloc[0]["time_slot"]
)
print(f"Actual Response Rate = {personal_model_X_test_scaled['prev_response_rate'].iloc[0]}")

    time_slot  average_response_probability
 late morning                          6.62
  mid morning                          5.92
      evening                          5.83
    afternoon                          5.74
        night                          3.45
early morning                          1.61

Recommended time slot: late morning
Actual Response Rate = -0.6416740058856


In [43]:
# --------------------------------------------------
# 1. Define time-slot columns
# --------------------------------------------------

TIME_SLOT_COLUMNS = [
    "time_slot_mid morning",
    "time_slot_late morning",
    "time_slot_afternoon",
    "time_slot_evening",
    "time_slot_night"
]

TIME_SLOTS = [
    "mid morning",
    "late morning",
    "afternoon",
    "evening",
    "night"
]

# --------------------------------------------------
# 2. Prepare test data
# --------------------------------------------------

# Copy test features
X_base = personal_model_X_test.copy().reset_index(drop=True)

uuid_test = test["uuid"].copy()
y_test = test["responded"].copy()
# Preserve user IDs and actual labels
users = np.asarray(uuid_test)
actual = np.asarray(y_test)

# Check alignment
assert len(X_base) == len(users) == len(actual)

# Ensure all time-slot columns exist
missing = set(TIME_SLOT_COLUMNS) - set(X_base.columns)

if missing:
    raise ValueError(f"Missing time-slot columns: {missing}")

# --------------------------------------------------
# 3. Generate five scenarios
# --------------------------------------------------

all_predictions = []

for slot, slot_column in zip(TIME_SLOTS, TIME_SLOT_COLUMNS):

    # Copy the original test features
    X_scenario = X_base.copy()

    # Set all time-slot columns to zero
    X_scenario[TIME_SLOT_COLUMNS] = 0

    # Activate the selected time slot
    X_scenario[slot_column] = 1

    # Predict response probabilities
    probabilities = personal_model.predict_proba(
        X_scenario
    )[:, 1]

    # Store predictions
    scenario_results = pd.DataFrame({
        "uuid": users,
        "responded": actual,
        "time_slot": slot,
        "predicted_probability": probabilities
    })

    all_predictions.append(scenario_results)

# --------------------------------------------------
# 4. Combine all scenarios
# --------------------------------------------------

test_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

print("Generated predictions successfully!")
print("Shape:", test_predictions.shape)

print(test_predictions.head(10))


Generated predictions successfully!
Shape: (113005, 4)
   uuid  responded    time_slot  predicted_probability
0  u247          0  mid morning               0.082213
1  u151          0  mid morning               0.118937
2  u295          0  mid morning               0.000047
3  u330          0  mid morning               0.000587
4  u146          0  mid morning               0.032527
5  u195          0  mid morning               0.981012
6  u144          0  mid morning               0.815116
7  u238          0  mid morning               0.867787
8  u084          0  mid morning               0.001650
9  u287          0  mid morning               0.338355


In [48]:
prediction_summary = (
    test_predictions
    .groupby("time_slot")["predicted_probability"]
    .agg(["mean", "std", "min", "max"])
)

print(prediction_summary)
print()

predicted_rates = (
    test_predictions
    .pivot_table(
        index="uuid",
        columns="time_slot",
        values="predicted_probability"
    )
)

print(predicted_rates.head(10))
print()

# Check probability distribution
print(
    test_predictions["predicted_probability"].describe()
)
print()

# Check how much probabilities vary across slots
print(
    predicted_rates.std(axis=1).describe()
)
print()

                  mean       std  min  max
time_slot                                 
afternoon     0.334726  0.410142  0.0  1.0
evening       0.335592  0.410419  0.0  1.0
late morning  0.343015  0.412730  0.0  1.0
mid morning   0.336535  0.410719  0.0  1.0
night         0.306271  0.400167  0.0  1.0

time_slot  afternoon   evening  late morning  mid morning     night
uuid                                                               
u000        0.877164  0.877697      0.882059     0.878272  0.856581
u002        0.479247  0.480066      0.487079     0.480959  0.452151
u003        0.081332  0.081939      0.087254     0.082604  0.063014
u004        0.574427  0.575675      0.586338     0.577034  0.532781
u005        0.414713  0.415469      0.421904     0.416291  0.389166
u006        0.999962  0.999963      0.999967     0.999963  0.999935
u007        0.200293  0.202087      0.217662     0.204047  0.144768
u008        0.275296  0.276801      0.289842     0.278445  0.227697
u009        0.9774

In [16]:
import pandas as pd

time_slots = [
    "early morning",
    "mid morning",
    "late morning",
    "afternoon",
    "evening",
    "night"
]

# Use the original dataset, before scaling.
train_original = training_dataset.iloc[:split_index].copy()
test_original = training_dataset.iloc[split_index:].copy()

# Recover each notification's actual time slot.
# Early morning is the omitted reference category.
def get_time_slot(row):
    for slot in time_slots[1:]:
        column = f"time_slot_{slot}"

        if row[column] == 1:
            return slot

    return "early morning"

train_original["time_slot"] = train_original.apply(
    get_time_slot, axis=1
)

test_original["time_slot"] = test_original.apply(
    get_time_slot, axis=1
)

# 1. Calculate response rates in the training data.
training_slot_rates = (
    train_original.groupby("time_slot")["responded"]
    .mean()
    .reindex(time_slots)
)

print("Training response rates:")
print((training_slot_rates * 100).round(2))

# 2. Choose the best slot using training data only.
best_baseline_slot = training_slot_rates.idxmax()

print("\nBest-time-slot baseline:", best_baseline_slot)

# 3. Evaluate notifications that actually occurred in that slot
# in the test data.
baseline_test_rows = test_original[
    test_original["time_slot"] == best_baseline_slot
]

if len(baseline_test_rows) > 0:
    baseline_test_rate = baseline_test_rows["responded"].mean()

    print("Test notifications in baseline slot:",
          len(baseline_test_rows))
    print("Observed test response rate:",
          round(baseline_test_rate * 100, 2), "%")
else:
    print("No test notifications were observed in this slot.")
print("original", round(train_original["responded"].mean() * 100, 2), "%")

Training response rates:
time_slot
early morning     3.92
mid morning      25.97
late morning     42.59
afternoon        42.62
evening          43.09
night            23.55
Name: responded, dtype: float64

Best-time-slot baseline: evening
Test notifications in baseline slot: 3051
Observed test response rate: 27.14 %
original 29.97 %


### Model for each user

In [38]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    confusion_matrix
)


# ==========================================
# 1. Load the dataset
# ==========================================

training_dataset = pd.read_csv("encoded_training_table.csv")


# Columns that will be scaled
scale_cols = [
    'battery_level',
    'light_level',
    'noise_level',
    'acc_avg',
    'notifications_30m',
    'app_events_30m',
    'unlocks_30m',
    'prev_notifications',
    'prev_response_rate',
    'recent_response_rate',
    'slot_prev_response_rate',
    'mins_since_prev_notification',
    'age'
]


# ==========================================
# 2. Create dictionaries to store models
# ==========================================

user_models = {}

user_scalers = {}

user_results = []


# ==========================================
# 3. Train and evaluate one model per user
# ==========================================

for user_id, user_data in training_dataset.groupby('uuid'):

    print(f"\nProcessing user: {user_id}")

    # Keep each user's observations in their existing order.
    # This assumes the CSV is already ordered chronologically
    # within each user.

    user_data = user_data.reset_index(drop=True)

    # --------------------------------------
    # 3.1 Separate features and target
    # --------------------------------------

    X = user_data.drop(columns=['uuid', 'responded'])
    y = user_data['responded']

    # --------------------------------------
    # 3.2 Split into training and testing
    # --------------------------------------

    split_index = int(len(user_data) * 0.80)

    X_train = X.iloc[:split_index].copy()
    X_test = X.iloc[split_index:].copy()

    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    # Skip users without enough training/testing data
    if len(X_train) < 2 or len(X_test) == 0:
        print("Skipped: Not enough observations.")
        continue

    # Logistic Regression needs both classes in training
    if y_train.nunique() < 2:
        print("Skipped: Training data contains only one target class.")
        continue

    # --------------------------------------
    # 3.3 Scale numerical features
    # --------------------------------------

    scaler = StandardScaler()

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    X_train_scaled[scale_cols] = scaler.fit_transform(
        X_train[scale_cols]
    )

    X_test_scaled[scale_cols] = scaler.transform(
        X_test[scale_cols]
    )

    # --------------------------------------
    # 3.4 Create and train the user's model
    # --------------------------------------

    user_model = LogisticRegression(
        max_iter=2000,
        random_state=42
    )

    user_model.fit(X_train_scaled, y_train)

    # Store this user's model and scaler
    user_models[user_id] = user_model
    user_scalers[user_id] = scaler

    # --------------------------------------
    # 3.5 Make predictions
    # --------------------------------------

    y_predictions = user_model.predict(X_test_scaled)

    y_probability = user_model.predict_proba(
        X_test_scaled
    )[:, 1]

    # --------------------------------------
    # 3.6 Evaluate the user's model
    # --------------------------------------

    accuracy = accuracy_score(y_test, y_predictions)

    precision = precision_score(
        y_test, y_predictions, zero_division=0
    )

    recall = recall_score(
        y_test, y_predictions, zero_division=0
    )

    f1 = f1_score(
        y_test, y_predictions, zero_division=0
    )

    # ROC-AUC requires both classes in the test data
    if y_test.nunique() == 2:
        roc_auc = roc_auc_score(y_test, y_probability)
    else:
        roc_auc = np.nan

    loss = log_loss(
        y_test,
        y_probability,
        labels=[0, 1]
    )

    brier = brier_score_loss(y_test, y_probability)

    cm = confusion_matrix(
        y_test,
        y_predictions,
        labels=[0, 1]
    )

    # --------------------------------------
    # 3.7 Store evaluation results
    # --------------------------------------

    user_results.append({
        'uuid': user_id,
        'training_samples': len(X_train),
        'testing_samples': len(X_test),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'log_loss': loss,
        'brier_score': brier,
        'confusion_matrix': cm.tolist()
    })

    print("Accuracy:", round(accuracy, 2))
    print("Precision:", round(precision, 2))
    print("Recall:", round(recall, 2))
    print("F1-score:", round(f1, 2))
    print("ROC-AUC:", round(roc_auc, 2) if not np.isnan(roc_auc) else "N/A")
    print("Log loss:", round(loss, 2))
    print("Brier score:", round(brier, 2))
    print("Confusion Matrix:")
    print(cm)


# ==========================================
# 4. Display individual user results
# ==========================================

results_df = pd.DataFrame(user_results)

print("\n==========================================")
print("INDIVIDUAL USER RESULTS")
print("==========================================")

print(results_df.to_string(index=False))


# ==========================================
# 5. Calculate average performance
# ==========================================

if not results_df.empty:

    metric_columns = [
        'accuracy',
        'precision',
        'recall',
        'f1_score',
        'roc_auc',
        'log_loss',
        'brier_score'
    ]

    average_results = results_df[metric_columns].mean()

    print("\n==========================================")
    print("AVERAGE PERFORMANCE ACROSS USERS")
    print("==========================================")

    print(average_results.round(3))

    print("\nNumber of users successfully trained:",
          len(user_models))

else:
    print("\nNo user models were successfully trained.")



Processing user: u000
Accuracy: 0.78
Precision: 0.0
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.74
Log loss: 0.43
Brier score: 0.15
Confusion Matrix:
[[25  2]
 [ 5  0]]

Processing user: u001
Accuracy: 0.81
Precision: 0.5
Recall: 0.44
F1-score: 0.47
ROC-AUC: 0.78
Log loss: 0.47
Brier score: 0.15
Confusion Matrix:
[[35  4]
 [ 5  4]]

Processing user: u002
Accuracy: 0.85
Precision: 1.0
Recall: 0.09
F1-score: 0.17
ROC-AUC: 0.69
Log loss: 0.44
Brier score: 0.13
Confusion Matrix:
[[54  0]
 [10  1]]

Processing user: u003
Accuracy: 0.98
Precision: 0.0
Recall: 0.0
F1-score: 0.0
ROC-AUC: N/A
Log loss: 0.13
Brier score: 0.03
Confusion Matrix:
[[80  2]
 [ 0  0]]

Processing user: u004
Accuracy: 0.7
Precision: 0.05
Recall: 0.5
F1-score: 0.08
ROC-AUC: 0.76
Log loss: 0.62
Brier score: 0.21
Confusion Matrix:
[[51 21]
 [ 1  1]]

Processing user: u005
Accuracy: 0.81
Precision: 0.67
Recall: 0.36
F1-score: 0.47
ROC-AUC: 0.86
Log loss: 0.42
Brier score: 0.13
Confusion Matrix:
[[35  2]
 [ 7  4]]

Processing use